In [1]:
# Load the right model that we want to analyze
import torch
from models.bert import ContinuityBERT
from train import parse_args
import create_knowledge_graph as kg_utils
from data import utils


def create_bert_model(config):
    encoder_type = config["encoder_type"]
    use_kg = "kg" in config["model_type"]
    model = ContinuityBERT(
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        n_gnn_layers=config["n_gnn_layers"],
        hidden_dim=config["hidden_dim"],
        input_dim=utils.SENTENCE_ENCODER_DIM[encoder_type],
        use_kg=use_kg,
        kg_node_dim=kg_utils.KG_NODE_DIM,
        kg_edge_dim=kg_utils.KG_EDGE_DIM,
        dropout=config["dropout"],
        gnn_type=config["gnn_type"],
    )
    return model

config_bert_kg_gat = {
    "train_ratio": 0.5,
    "batch_size": 64,
    "n_continuity_errors": 1, #[1, 2
    "n_heads": 8,
    "n_layers": 3,
    "n_gnn_layers": 2,
    "hidden_dim": 20,
    "dropout": 0.2,
    "n_epochs": 100,
    "n_runs": 5,
    "lr": 1e-5,
    "pr_threshold": 0.3,
    "encoder_type": "all-MiniLM-L6-v2",
    "gnn_type": "gatv2", #["gatv2", "gcn"],
    "model_type": "bert_kg"
}

model = create_bert_model(config_bert_kg_gat)

initialized continuityBERT with 628261 parameters.


In [2]:
# Load saved weights into 
MODEL_WEIGHTS_PATH = "./results/bert_kg_gat/bert_kg_gat-1_error-params.pkl"
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH))

<All keys matched successfully>

In [3]:
# Load example data
import pickle as pkl
input_data = "./data/dataset/encoded/test/test_1_error.pkl"
with open(input_data, "rb") as f:
    dataset, _ = pkl.load(f)

xs, ys = [], []
kgs = []
docs = []
for i, (x, y, kg, doc) in enumerate(dataset):
    xs.append(x)
    ys.append(y)
    kgs.append(kg)
    docs.append(doc)

In [4]:
example_datapoint = 0
x = xs[example_datapoint]
y = ys[example_datapoint]
kg = kgs[example_datapoint]

print(f"data_{example_datapoint} # sentences: {len(x)}")

data_0 # sentences: 126


In [8]:
y_hat = model.forward(x.reshape([1, len(x), -1]), [kg])
print(y_hat)

tensor([[2.0879e-22, 9.3010e-23, 7.2476e-23, 1.0000e+00, 1.7440e-22, 2.5330e-22,
         8.2413e-20, 1.0832e-22, 1.4584e-22, 9.6879e-23, 8.6093e-23, 1.1503e-22,
         1.2074e-20, 1.5056e-22, 7.6804e-23, 1.7802e-22, 5.3301e-23, 1.2462e-22,
         9.9501e-23, 1.7545e-22, 2.1438e-22, 2.1656e-22, 1.4267e-22, 9.4278e-23,
         1.2117e-22, 1.3297e-22, 8.8477e-23, 2.4470e-22, 7.0376e-23, 1.2370e-22,
         7.2083e-23, 1.0424e-22, 1.2141e-22, 1.5614e-22, 1.6505e-22, 1.7653e-22,
         1.6946e-22, 2.1423e-22, 1.5682e-22, 1.7955e-22, 2.5140e-22, 7.1828e-23,
         3.1294e-22, 2.7990e-22, 1.0034e-22, 2.3578e-22, 1.0634e-22, 2.2545e-22,
         1.1749e-22, 1.0986e-22, 1.7895e-22, 2.2054e-22, 2.0631e-22, 7.4190e-23,
         5.8760e-23, 1.5688e-22, 1.7437e-22, 7.3047e-23, 1.8093e-22, 1.2165e-22,
         2.5318e-22, 1.2315e-22, 1.1104e-22, 1.1394e-22, 1.1322e-22, 1.3221e-22,
         1.1733e-22, 1.1458e-22, 1.9997e-22, 3.9662e-22, 1.2226e-22, 1.3621e-22,
         4.9788e-22, 2.2291e

In [10]:
len(y_hat[0])

126